In [1]:
%pip install fake-useragent

Note: you may need to restart the kernel to use updated packages.


In [11]:
%pip install selenium

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.7 MB 3.2 MB/s eta 0:00:03
   ------- -------------------------------- 1.8/9.7 MB 3.9 MB/s eta 0:00:03
   ---------- ----------------------------- 2.6/9.7 MB 4.1 MB/s eta 0:00:02
   -------------- ------------------------- 3.4/9.7 MB 4.0 MB/s eta 0:00:02
   ----------------- ---------------------- 4.2/9.7 MB 3.8 MB/s eta 0:00:02
   ------------------- -------------------- 4.7/9.7 MB 3.6 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.7 MB 3.4 MB/s eta 0:00:02
   ------------------------ --------------- 6.0/9.7 MB 3.5 MB/s eta 0:00:02
   ---------------------------- ----------- 6.8/9.7 MB 3.5 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.7 MB 3.5 MB/s eta 0:00:01
   ---------------------------------- ----- 8.4/9.7 MB 3.5 MB/s eta 0:00:01
   -----------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
import requests
import time
import random
import pandas as pd
import json
import os
import re
from datetime import datetime
from fake_useragent import UserAgent
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


CATEGORIES = [
    "платье вечернее", "платье повседневное", "платье летнее", "платье офисное",
    "пальто женское", "куртка женская", "пуховик женский",
    "блузка женская", "рубашка женская",
    "свитер женский", "кардиган женский", "водолазка женская",
    "юбка женская", "юбка мини", "юбка макси",
    "джинсы женские", "брюки женские", "брюки спортивные женские",
    "костюм женский", "пиджак женский", "жакет женский",
    "футболка женская", "толстовка женская", "худи женское",
    "нижнее белье женское",
    "куртка мужская", "пальто мужское", "пуховик мужской",
    "рубашка мужская", "рубашка поло мужская",
    "свитер мужской", "водолазка мужская",
    "джинсы мужские", "брюки мужские", "брюки спортивные мужские",
    "костюм мужской", "пиджак мужской",
    "футболка мужская", "толстовка мужская", "худи мужское",
    "спортивный костюм", "ветровка",
]

PAGES_PER_CATEGORY = 4
PRODUCTS_PER_PAGE = 30
MIN_DELAY = 3.0
MAX_DELAY = 5.0
MAX_RETRIES = 3
PRODUCTS_FILE = "wb_products_full.csv"
BACKUP_DIR = "wb_backups"
os.makedirs(BACKUP_DIR, exist_ok=True)


chrome_options = Options()
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)
chrome_options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36')


class WildberriesParser:

    def __init__(self):
        self.ua = UserAgent()
        self.search_url = "https://www.wildberries.ru/__internal/u-search/exactmatch/ru/common/v18/search"
        self.session = requests.Session()
        self.cookies = {
            '_wbauid': '1542039911763969751',
            'x_wbaas_token': '1.1000.d33dffe5b1194bc39776f55e96429aee.MHwzNy43Ni4xNTkuNTR8TW96aWxsYS81LjAgKFdpbmRvd3MgTlQgMTAuMDsgV2luNjQ7IHg2NCkgQXBwbGVXZWJLaXQvNTM3LjM2IChLSFRNTCwgbGlrZSBHZWNrbykgQ2hyb21lLzE0Ni4wLjAuMCBZYUJyb3dzZXIvMjYuNC4wLjAgU2FmYXJpLzUzNy4zNnwxNzgxNjAyMDA0fHJldXNhYmxlfDJ8ZXlKb1lYTm9Jam9pSW4wPXwwfDN8MTc4MTQ3MjQwNHwx.MEUCIDjN/l984esqOqBPoFqGY8V1rWehkKQOlD10ALrWUgOPAiEA7zor0yC8Uy+oSOOeJokZRZ871ynkjbd1Lcw94p7GKiQ=',
            '_cp': '1',
        }
        self.x_spa_version = '14.13.6'
        self.device_id = 'site_2633732a97854ab2bcdbae007471feb4'
        self.driver = webdriver.Chrome(options=chrome_options)
        self.driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        self.total_products = 0
        self.total_descriptions = 0
        self.total_reviews = 0

    def _headers(self):
        return {
            'accept': '*/*', 'accept-language': 'ru,en;q=0.9',
            'deviceid': self.device_id, 'priority': 'u=1, i',
            'referer': 'https://www.wildberries.ru/',
            'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "YaBrowser";v="26.4", "Yowser";v="2.5"',
            'sec-ch-ua-mobile': '?0', 'sec-ch-ua-platform': '"Windows"',
            'sec-fetch-dest': 'empty', 'sec-fetch-mode': 'cors',
            'sec-fetch-site': 'same-origin',
            'user-agent': self.ua.random,
            'x-requested-with': 'XMLHttpRequest',
            'x-spa-version': self.x_spa_version,
            'x-userid': '0',
        }

    def _request(self, url, params=None):
        for attempt in range(MAX_RETRIES):
            try:
                if params:
                    resp = self.session.get(url, params=params, headers=self._headers(), cookies=self.cookies, timeout=30)
                else:
                    resp = self.session.get(url, headers=self._headers(), cookies=self.cookies, timeout=30)
                if resp.status_code == 429:
                    wait = int(resp.headers.get('Retry-After', 10))
                    time.sleep(wait)
                    continue
                if resp.status_code in (404, 498):
                    return None
                resp.raise_for_status()
                return resp
            except:
                if attempt < MAX_RETRIES - 1:
                    time.sleep((attempt + 1) * 2)
        return None

    def search_page(self, query, page):
        params = {
            "ab_testing": "false", "appType": "1", "curr": "rub",
            "dest": "-1257786", "hide_dtype": "9;11;15",
            "hide_vflags": "4294967296", "inheritFilters": "false",
            "lang": "ru", "locale": "ru", "query": query,
            "resultset": "catalog", "sort": "popular", "spp": "30",
            "suppressSpellcheck": "false", "page": str(page),
            "limit": str(PRODUCTS_PER_PAGE),
        }
        resp = self._request(self.search_url, params)
        if not resp:
            return []
        try:
            data = resp.json()
        except:
            return []
        products_raw = data.get('products', [])
        results = []
        for p in products_raw:
            product_id = p.get('id')
            brand = p.get('brand', '')
            name = p.get('name', '')
            full_name = f"{brand} {name}".strip() if brand else name
            sizes = p.get('sizes', [])
            basic_price = sizes[0].get('price', {}).get('basic', 0) if sizes else 0
            product_price = sizes[0].get('price', {}).get('product', 0) if sizes else 0
            discount_pct = round((1 - product_price / basic_price) * 100) if basic_price and product_price and basic_price > product_price else 0
            sizes_names = [s.get('name', '') for s in sizes if s.get('name')]
            colors_list = p.get('colors', [])
            colors_names = [c.get('name', '') for c in colors_list if c.get('name')]
            results.append({
                'product_id': product_id,
                'imt_id': p.get('root', product_id),
                'name': full_name, 'brand': brand,
                'price': product_price / 100 if product_price else 0,
                'price_no_discount': basic_price / 100 if basic_price else 0,
                'discount_pct': discount_pct,
                'rating': p.get('reviewRating', 0) or p.get('rating', 0),
                'reviews_count': p.get('feedbacks', 0) or 0,
                'subject_id': p.get('subjectId', ''),
                'supplier_rating': p.get('supplierRating', 0),
                'sizes': ', '.join(sizes_names) if sizes_names else '',
                'colors': ', '.join(colors_names) if colors_names else '',
                'category_query': query, 'page': page,
                'collected_at': datetime.now().isoformat()
            })
        return results

    def get_details(self, product_id):
        url = f"https://www.wildberries.ru/catalog/{product_id}/detail.aspx"
        result = {
            'description': '',
            'composition': '', 'color': '', 'gender': '',
            'country': '', 'care': '',
            'ai_summary': '', 'reviews_text': '',
        }
        try:
            self.driver.get(url)
            time.sleep(3)

            try:
                btn = WebDriverWait(self.driver, 10).until(
                    EC.element_to_be_clickable((By.CLASS_NAME, "btnDetailText--nrkiv"))
                )
                self.driver.execute_script("arguments[0].scrollIntoView(true);", btn)
                time.sleep(1)
                btn.click()
                time.sleep(2)
            except:
                pass

            try:
                desc = WebDriverWait(self.driver, 5).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "#section-description p"))
                )
                result['description'] = desc.text.strip()[:3000]
            except:
                pass

            try:
                rows = self.driver.find_elements(By.CSS_SELECTOR, ".content--zb_r9 tr")
                for row in rows:
                    try:
                        th = row.find_element(By.TAG_NAME, "th").text.strip().lower()
                        td = row.find_element(By.TAG_NAME, "td").text.strip()
                        if 'состав' in th: result['composition'] = td
                        elif 'цвет' in th: result['color'] = td
                        elif 'пол' in th: result['gender'] = td
                        elif 'страна' in th: result['country'] = td
                        elif 'уход' in th: result['care'] = td
                    except:
                        pass
            except:
                pass

            try:
                reviews_btn = self.driver.find_element(By.CSS_SELECTOR, "a.comments__btn-all")
                reviews_url = reviews_btn.get_attribute("href")
                self.driver.get(reviews_url)
                time.sleep(3)
            except:
                pass

            try:
                ai = self.driver.find_element(By.CLASS_NAME, "generative-feedback__text")
                result['ai_summary'] = ai.text.strip()
            except:
                pass

            try:
                items = self.driver.find_elements(By.CSS_SELECTOR, "li.comments__item")
                texts = []
                for item in items[:5]:
                    try:
                        t = item.find_element(By.CLASS_NAME, "feedback__text")
                        if t:
                            texts.append(t.text.strip())
                    except:
                        pass
                result['reviews_text'] = ' ||| '.join(texts)
            except:
                pass

        except:
            pass
        return result

    def collect_category(self, query):
        cat_products = []
        seen_ids = set()

        for page in range(1, PAGES_PER_CATEGORY + 1):
            products = self.search_page(query, page)
            if not products:
                break

            for prod in products:
                pid = prod['product_id']
                if pid in seen_ids:
                    continue
                seen_ids.add(pid)

                details = self.get_details(pid)
                prod.update(details)

                self.total_products += 1
                if prod['description']:
                    self.total_descriptions += 1
                if prod['reviews_text']:
                    self.total_reviews += 1

                cat_products.append(prod)
                time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

        safe_name = re.sub(r'\s+', '_', query)
        if cat_products:
            pd.DataFrame(cat_products).to_csv(
                f"{BACKUP_DIR}/prod_{safe_name}.csv", index=False, encoding='utf-8-sig'
            )

        return pd.DataFrame(cat_products)

    def run(self):
        all_data = []

        print(f"Начало сбора: {len(CATEGORIES)} категорий, по {PAGES_PER_CATEGORY} страниц")

        for i, query in enumerate(CATEGORIES):
            df_cat = self.collect_category(query)
            if not df_cat.empty:
                all_data.append(df_cat)

            print(f"[{i+1}/{len(CATEGORIES)}] {query}: товаров {len(df_cat)}, "
                  f"всего {self.total_products}, опис {self.total_descriptions}, отз {self.total_reviews}")

            if i < len(CATEGORIES) - 1:
                time.sleep(random.uniform(2, 4))

        full_df = pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()
        if not full_df.empty:
            full_df = full_df.drop_duplicates(subset='product_id')

        full_df.to_csv(PRODUCTS_FILE, index=False, encoding='utf-8-sig')

        print(f"\nСбор завершен")
        print(f"Всего товаров: {len(full_df)}")
        print(f"С описанием: {(full_df['description'] != '').sum()}")
        print(f"С AI-сводкой: {(full_df['ai_summary'] != '').sum()}")
        print(f"С отзывами: {(full_df['reviews_text'] != '').sum()}")
        print(f"Файл: {PRODUCTS_FILE}")

        self.driver.quit()
        return full_df


parser = WildberriesParser()
df = parser.run()

Начало сбора: 42 категорий, по 4 страниц
[1/42] платье вечернее: товаров 113, всего 113, опис 112, отз 106
[2/42] платье повседневное: товаров 115, всего 228, опис 227, отз 216
[3/42] платье летнее: товаров 116, всего 344, опис 343, отз 328
[4/42] платье офисное: товаров 117, всего 461, опис 459, отз 440
[5/42] пальто женское: товаров 110, всего 571, опис 566, отз 545
[6/42] куртка женская: товаров 119, всего 690, опис 685, отз 659
[7/42] пуховик женский: товаров 119, всего 809, опис 804, отз 776
[8/42] блузка женская: товаров 114, всего 923, опис 916, отз 884
[9/42] рубашка женская: товаров 114, всего 1037, опис 1030, отз 989
[10/42] свитер женский: товаров 118, всего 1155, опис 1148, отз 1101
[11/42] кардиган женский: товаров 120, всего 1275, опис 1266, отз 1220
[12/42] водолазка женская: товаров 117, всего 1392, опис 1383, отз 1334
[13/42] юбка женская: товаров 112, всего 1504, опис 1494, отз 1441
[14/42] юбка мини: товаров 116, всего 1620, опис 1609, отз 1546
[15/42] юбка макси: то

KeyboardInterrupt: 